<a href="https://colab.research.google.com/github/4nkit-3isGGS/first-ai-agent-using-adk/blob/main/Multi_agent_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🚀 Multi-Agent Systems & Workflow Patterns

In the previous notebook [https://github.com/4nkit-3isGGS/first-ai-agent-using-adk/blob/main/first_ai_agent.ipynb], you built a single agent that could take action. Now, you'll learn how to scale up by building agent teams.

Just like a team of people, you can create specialized agents that collaborate to solve complex problems. This is called a multi-agent system, and it's one of the most powerful concepts in AI agent development.

In this notebook, you'll:

✅ Learn when to use multi-agent systems in Agent Development Kit (ADK)

✅ Build your first system using an LLM as a "manager"

✅ Learn three core workflow patterns (Sequential, Parallel, and Loop) to coordinate your agent teams

Just like the previous notebook, you will first Setup before proceeding futher. To go through with the setup processes visit [https://github.com/4nkit-3isGGS/first-ai-agent-using-adk/blob/main/first_ai_agent.ipynb] and see

Section 1: Setup
1.1: Install dependencies
1.2: Configure your Gemini API Key
1.3:  Import ADK components
1.4: Configure Retry Options

In [1]:
pip install google-adk

In [5]:
import os
from google.colab import userdata

try:
  GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
  os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
  print("✅ Gemini API key setup complete.")

except Exception as e:
     print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Colab secrets. Details: {e}")


✅ Gemini API key setup complete.


In [6]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [8]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)
print("Done!")

Done!


🤔 Section 2: Why Multi-Agent Systems? + Your First Multi-Agent

The Problem: The "Do-It-All" Agent

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

The Solution: A Team of Specialists

Instead of one "do-it-all" agent, we can build a multi-agent system. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent only does research, another only writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

Architecture: Single Agent vs Multi-Agent Team

2.1 Example: Research & Summarization System
Let's build a system with two specialized agents:

Research Agent - Searches for information using Google Search

Summarizer Agent - Creates concise summaries from research findings

In [9]:
try:
  research_agent = Agent(
    name= "ResearchAgent",
    model= Gemini(
        model= "gemini-2.5-flash-lite",
        retry_options= retry_config
        ),
    instruction=""" You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.
    """,
    tools=[google_search],
    output_key= "research_findings" # Storing the result of this agent in {research_findings} key.
    )
  print("Research Agent created successfully.")
except Exception as e:
  print(f"Error creating Research Agent: {e}")



Research Agent created successfully.


In [10]:
try:
  summary_agent = Agent(
    name= "SummaryAgent",
    model= Gemini(
         model= "gemini-2.5-flash-lite",
         retry_options= retry_config
    ),
    instruction= """Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.
    """,
    output_key="final_summary",

  )
  print("Summary Agent created Successfully!")
except Exception as e:
  print(f"Error creating Summary Agent: {e}")


Summary Agent created Successfully!


Then we bring the agents together under a root agent, or coordinator: